In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [15]:
# =========================
# 1. Imports
# =========================
import os, json
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# =========================
# 2. Load JSONL (ROBUST)
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/primevul-dataset26"

def load_jsonl(file):
    data = []
    with open(file, "r") as f:
        for line in f:
            obj = json.loads(line)

            # robust field detection
            code = (
                obj.get("func_before") or
                obj.get("code") or
                obj.get("func") or
                obj.get("before") or
                ""
            )

            label = obj.get("target", obj.get("label", 0))

            if code.strip() != "":
                data.append((code, int(label)))

    return data

train_data = load_jsonl(os.path.join(base_path, "primevul_train_paired.jsonl"))
val_data   = load_jsonl(os.path.join(base_path, "primevul_valid_paired.jsonl"))
test_data  = load_jsonl(os.path.join(base_path, "primevul_test_paired.jsonl"))

# merge train + val
all_data = train_data + val_data

texts = [x[0] for x in all_data]
labels = [x[1] for x in all_data]

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.1, stratify=labels, random_state=42
)

X_test = [x[0] for x in test_data]
y_test = [x[1] for x in test_data]

print("Train size:", len(X_train))

# =========================
# 3. Character Encoding (STABLE)
# =========================
all_text = "".join(X_train)
chars = sorted(list(set(all_text)))

char2idx = {c: i+1 for i, c in enumerate(chars)}
char2idx["<PAD>"] = 0

def encode(text):
    return [char2idx.get(c, 0) for c in text]

MAX_LEN = 300   # shorter = better for LSTM

def pad(seq):
    return seq[:MAX_LEN] + [0]*(MAX_LEN - len(seq))

X_train = [pad(encode(t)) for t in X_train]
X_val   = [pad(encode(t)) for t in X_val]
X_test  = [pad(encode(t)) for t in X_test]

# =========================
# 4. Dataset
# =========================
class VulDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(VulDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(VulDataset(X_val, y_val), batch_size=64)
test_loader  = DataLoader(VulDataset(X_test, y_test), batch_size=64)

# =========================
# 5. LSTM Model
# =========================
class CharLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 128, num_layers=2, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = hidden[-1]
        return self.fc(self.dropout(out)).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CharLSTM(len(char2idx)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# =========================
# 6. Training
# =========================
for epoch in range(8):
    model.train()
    total_loss = 0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

# =========================
# 7. Evaluation
# =========================
model.eval()
preds, true = [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        out = torch.sigmoid(model(Xb.to(device)))
        preds.extend((out > 0.5).cpu().numpy())
        true.extend(yb.numpy())

print("\nAccuracy:", accuracy_score(true, preds))
print("\nClassification Report:\n", classification_report(true, preds))

Train size: 7684
Epoch 1 Loss: 0.6944
Epoch 2 Loss: 0.6937
Epoch 3 Loss: 0.6937
Epoch 4 Loss: 0.6933
Epoch 5 Loss: 0.6927
Epoch 6 Loss: 0.6927
Epoch 7 Loss: 0.6926
Epoch 8 Loss: 0.6921

Accuracy: 0.5011494252873563

Classification Report:
               precision    recall  f1-score   support

         0.0       0.50      0.66      0.57       435
         1.0       0.50      0.35      0.41       435

    accuracy                           0.50       870
   macro avg       0.50      0.50      0.49       870
weighted avg       0.50      0.50      0.49       870

